
# AELIONIX BLACKFORGE -- Phase 14.1 Colab Validation

This notebook performs a deterministic, one-click validation of the **Real
Mission Integration, LLM Orchestration & Development Console** (Phase 14.1).

Phase 14.1 introduces a **bounded mission orchestration layer** with explicit
**execution modes** (MOCK / REAL / AUTO), evidence-backed **findings**, and a
token-gated **Development Console** reachable over a temporary **public URL**.
It is an *authorized, read-only, evidence-gathering and planning* layer -- not an
attack engine and not an exploitation system.

The notebook validates, end-to-end and in order:

* **reusable mission + scope** -- a mission is created from a validated seed
  target and confined to a bounded `TargetScope` with an explicit assessment
  profile and execution budget
* **deterministic orchestrator** -- the `MissionOrchestrator` is the only
  component allowed to decide whether a planner proposal executes; every
  dispatch passes registration, authorization, scope, adapter and typing gates
* **execution modes, enforced** -- REAL requires a real transport and FAILS
  (never silently falls back to mock); AUTO explicitly reports every fallback;
  MOCK forbids real transports
* **real-controlled, read-only observation** -- bounded std-lib DNS / TLS / HTTP
  metadata adapters fire only when execution mode + policy + profile allow;
  their output is redaction-safe, and their provenance is tagged `REAL`
* **evidence -> memory -> world model -> attack graph -> findings** -- every
  executed instruction normalizes into evidence, memory, world-model entities,
  a rebuilt descriptive attack graph, and deterministic evidence-backed findings
* **deterministic stop conditions** -- max steps, max capability calls, max
  runtime, max replans, repeated invalid plans, duplicate eviction, evidence
  saturation, REAL_MODE_GUARD, explicit cancellation, or a planner-requested stop
* **real Qwen planner** -- a real `Qwen/Qwen2.5-3B-Instruct` model drives the
  planner; the runtime records `invocation=REAL` only when the real model answered
* **Development Console over a public tunnel** -- an access-gated HTTP console
  whose real temporary public URL is printed and health-probed

> Run all cells top-to-bottom on Colab (internet + at least a CPU is required;
> the real-model cell benefits from a GPU runtime). The notebook fails loudly
> on any check.


---


In [ ]:

import sys
import platform

print("Blackforge Phase 14.1 Colab Validation (Mission Orchestration & Development Console)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")


---


In [ ]:

REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
from pathlib import Path
REPO_DIR = Path("/content/blackforge")
import subprocess, shutil, os

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")


---


In [ ]:

import subprocess
try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("Commit:", commit)
except Exception as e:
    print("Commit unavailable (expected in scratch checkouts):", e)


---


In [ ]:

!pip install hatchling --quiet
!pip install -e ".[dev]" --quiet
!pip install pyngrok --quiet


---


In [ ]:

import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.capabilities.registry",
    "blackforge.capabilities.models",
    "blackforge.capabilities.interface",
    "blackforge.authorization",
    "blackforge.scope.models",
    "blackforge.evidence.models",
    "blackforge.evidence.store",
    "blackforge.world_model.models",
    "blackforge.world_model.materializer",
    "blackforge.attack_graph.models",
    "blackforge.attack_graph.builder",
    "blackforge.orchestration",
    "blackforge.orchestration.models",
    "blackforge.orchestration.orchestrator",
    "blackforge.orchestration.planner",
    "blackforge.orchestration.routing",
    "blackforge.orchestration.adapters",
    "blackforge.ui",
    "blackforge.ui.access",
    "blackforge.ui.service",
    "blackforge.ui.console",
    "blackforge.ui.server",
    "blackforge.runtime.tunnel",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} -- {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Phase 14.1 module imports OK ({len(modules)} modules).")
print("Orchestration, real-adapter, tunnel, console-server imports: PASS")


---


In [ ]:

import subprocess, sys, os

print("Running automated test suite (Layer A deterministic + regression)...")
_test_env = {k: v for k, v in os.environ.items() if not k.startswith('BLACKFORGE_')}
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR), env=_test_env,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite (deterministic): PASS")


---


In [ ]:

import os
from pathlib import Path

DBROOT = Path("data/phase14_1_colab").resolve()
DBROOT.mkdir(parents=True, exist_ok=True)
os.environ["BLACKFORGE_DB_PATH"] = str(DBROOT / "blackforge.db")
os.environ["BLACKFORGE_MEMORY_DB_PATH"] = str(DBROOT / "memory.db")
os.environ["BLACKFORGE_EVIDENCE_DB_PATH"] = str(DBROOT / "evidence.db")
os.environ["BLACKFORGE_WORLD_MODEL_DB_PATH"] = str(DBROOT / "world_model.db")
for _p in (DBROOT / "evidence.db", DBROOT / "world_model.db"):
    _p.unlink(missing_ok=True)

from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
assert len(app.capability_registry.list_capabilities()) == 105
print("Bootstrap: PASS (105 registered capabilities)")


---


In [ ]:

# -- Orchestrator + scope: a mission is a bounded, authorized unit -------------
from blackforge.orchestration.models import (
    AdapterMode, AssessmentProfile, EvidenceSource, ExecutionMode, ExecutionPhase,
    InvestigationStatus, MissionSetup, MissionPolicy, StopReason,
)
from blackforge.orchestration.orchestrator import MissionOrchestrator, OrchestrationError

orch = MissionOrchestrator(app)

SEED = "getaelionix.com"
setup = MissionSetup(
    name="Colab Validation Mission",
    objective="authorized, read-only organizational assessment",
    seed_target=SEED,
    profile=AssessmentProfile.AUTHORIZED_ASSESSMENT,
    policy=MissionPolicy(
        max_steps=6,
        max_capability_calls=20,
        max_runtime_seconds=120.0,
        max_replans=3,
        allow_real_controlled=False,
        execution_mode=ExecutionMode.AUTO,
        planner_mode=ExecutionMode.AUTO,
    ),
)

mission = orch.create_mission(setup)
MID = str(mission.id)
scope = orch.scope_for(MID)

# Scope is bounded to the seed target and caps risk for the profile.
assert scope.allowed_targets and scope.allowed_targets[0].value == SEED
assert scope.allowed_capabilities == []
assert scope.max_risk_level.value in ("medium", "high")
assert mission.status.value in ("ready", "created")

# The policy records the declared mode; execution never upgrades or downgrades it.
assert orch.policy_for(MID).execution_mode is ExecutionMode.AUTO

print("Mission created:", MID, "| scope bounded to:", SEED)
print("Scope bounds + explicit execution mode: PASS")


---


In [ ]:

# -- Fail-closed validation: unknown capability & empty seed -------------------
from blackforge.scope.models import Target, TargetScope, ExecutionLimits
from blackforge.core.types import TargetType, RiskLevel

try:
    orch.create_mission(MissionSetup(name="n", objective="o", seed_target="  "))
    raise AssertionError("empty seed should have failed")
except OrchestrationError as e:
    print("Empty seed rejected (fail closed):", str(e))

limited = TargetScope(
    mission_id=MID,
    allowed_targets=[Target(value=SEED, target_type=TargetType.DOMAIN)],
    allowed_capabilities=["recon.dns"],
    max_risk_level=RiskLevel.LOW,
    execution_limits=ExecutionLimits(timeout_seconds=60),
)

view = orch.capability_view(MID)
print("Applicable capabilities for", SEED, ":", view.applicable)
assert view.applicable > 0
print("Fail-closed guardrails: PASS")


---


In [ ]:

# -- Planner: routing view, adapter modes and real-controlled surface -----------
cap_view = orch.capability_view(MID)
print(f"Registered={cap_view.registered} applicable={cap_view.applicable} "
      f"authorized={cap_view.authorized} real_controlled={cap_view.real_controlled}")
assert cap_view.registered == 105
assert cap_view.applicable > 0

# The four real-controlled adapters are installed but fire only when execution
# mode + policy + profile approve a real transport.
real_rows = [r for r in cap_view.rows if r.adapter is AdapterMode.REAL_CONTROLLED]
real_names = sorted(r.capability for r in real_rows)
print("Real-controlled adapters:", real_names)
assert real_names == [
    "recon.dns", "recon.http_metadata", "recon.tls_metadata",
    "webapi.security_header_analysis",
]
print("Planner/routing + adapter-mode surface: PASS")


---


In [ ]:

# -- Run a bounded mission with the deterministic rule-based planner ------------
#   execution_mode=AUTO + allow_real_controlled=False  =>  mock transport only.
state = orch.run_mission(MID, planner=orch.rule_planner)
print("Phase:", state.phase.value, "| stop:", state.stop_reason.value)
print("Steps completed:", state.steps_completed)
print("Capability calls:", state.capability_calls)
print(f"Mock observations: {state.mock_observations} | Real: {state.real_observations}")
print(f"Evidence rows after run: {len(orch.evidence_rows(MID))}")

assert state.steps_completed >= 1
assert state.stop_reason in (StopReason.MAX_STEPS, StopReason.NO_ACTIONS_REMAIN,
                             StopReason.MAX_CAPABILITY_CALLS, StopReason.EVIDENCE_SATURATION,
                             StopReason.MAX_REPLANS)
assert state.mock_observations >= 1
assert state.real_observations == 0
assert state.transport_mode != "real_controlled"
assert all(r.status is InvestigationStatus.COMPLETED for r in state.instructions)
for rec in state.instructions:
    assert rec.capability and rec.target and rec.reason
    assert rec.source.value in ("rule", "mock", "llm")
    assert rec.evidence_ids, f"{rec.capability} produced no evidence"

# Mock evidence rows carry MOCK provenance; findings are derived, never
# self-validated.
_mock_rows = orch.evidence_rows(MID)
assert _mock_rows and all(r.source is EvidenceSource.MOCK for r in _mock_rows)
_mock_evidence_statuses = [r.status for r in _mock_rows]
for _f in orch.findings(MID):
    assert _f.status in ("observed", "inferred", "hypothesized", "validated")
    # a finding can never validate itself: "validated" requires an evidence
    # row that actually reached validated status
    assert _f.status != "validated" or "validated" in _mock_evidence_statuses
print("Orchestration loop (deterministic, evidence-producing): PASS")


---


In [ ]:

# -- Evidence, assets (world model) and attack graph all materialize -----------
evidence_rows = orch.evidence_rows(MID)
assets = orch.assets(MID)
graph = orch.graph_summary(MID)

print("Evidence:", len(evidence_rows))
print("Assets:", [(a.name, a.classification.value) for a in assets][:6])
print("Graph nodes:", graph.node_count, "edges:", graph.edge_count,
      "open:", graph.open_path_count, "uncertain:", graph.uncertain_path_count)

assert evidence_rows, "no evidence was produced"
for asset in assets:
    assert asset.classification.value in ("in_scope", "candidate", "out_of_scope", "third_party", "unknown")
    assert isinstance(asset.authorized, bool)
print("Evidence -> world model -> attack graph materialization: PASS")


---


In [ ]:

# -- Deterministic stop conditions & cancellation ------------------------------
mission2 = orch.create_mission(MissionSetup(
    name="stop-demo", objective="stop-condition demo", seed_target=SEED,
    policy=MissionPolicy(max_steps=2, max_runtime_seconds=60.0),
))
state2 = orch.run_mission(str(mission2.id), planner=orch.rule_planner)
assert state2.stop_reason is StopReason.MAX_STEPS
print("MAX_STEPS stop condition: PASS ->", state2.stop_reason.value)

mission3 = orch.create_mission(MissionSetup(
    name="cancel-demo", objective="cancel demo", seed_target=SEED,
    policy=MissionPolicy(max_steps=10, max_runtime_seconds=60.0),
))
state3 = orch.cancel(str(mission3.id))
assert state3.phase is ExecutionPhase.CANCELLED
assert state3.stop_reason is StopReason.CANCELLED
print("CANCELLED stop condition: PASS")


---


In [ ]:

# -- Fail-closed planner: unknown capabilities stop the loop --------------------
from blackforge.orchestration.planner import RuleBasedPlanner, PlannerInvalidDecision
from blackforge.orchestration.models import DecisionKind, PlannerDecision, PlannerSource

class EmptyPlanner(RuleBasedPlanner):
    def plan(self, ctx):
        return PlannerDecision(
            kind=DecisionKind.INVESTIGATE, capability="__not_registered__",
            target=SEED, reason="should fail", source=PlannerSource.MOCK,
        )

mission4 = orch.create_mission(MissionSetup(
    name="closed-planner", objective="closed", seed_target=SEED,
    policy=MissionPolicy(max_steps=3, max_replans=2, max_capability_calls=10),
))
state4 = orch.run_mission(str(mission4.id), planner=EmptyPlanner())
# Repeated invalid plans fail closed to a deterministic stop.
assert state4.stop_reason in (
    StopReason.REPEATED_INVALID_PLANNER,
    StopReason.MAX_REPLANS,
)
print("Fail-closed planner (unknown capability): PASS ->", state4.stop_reason.value)


---


In [ ]:

# -- REAL execution without a real transport FAILS (no silent mock fallback) ---
#   execution_mode=REAL + real transport disabled  =>  REAL_MODE_GUARD.
mission_rg = orch.create_mission(MissionSetup(
    name="real-guard", objective="real guard demo", seed_target=SEED,
    profile=AssessmentProfile.AUTHORIZED_ASSESSMENT,
    policy=MissionPolicy(
        max_steps=1, max_runtime_seconds=60.0,
        allow_real_controlled=True,  # mission policy asks for real
        execution_mode=ExecutionMode.REAL,
        planner_mode=ExecutionMode.AUTO,
    ),
))
mid_rg = str(mission_rg.id)
orch.scope_for(mid_rg).allowed_capabilities = ["recon.dns"]
state_rg = orch.run_mission(mid_rg, planner=orch.rule_planner)
assert state_rg.stop_reason is StopReason.REAL_MODE_GUARD
assert state_rg.real_observations == 0 and state_rg.mock_observations == 0
# The run-state phase finalizes as COMPLETED; the failure is recorded on the
# mission record (MissionStatus.FAILED) and in the stop_reason.
assert str(orch.mission_record(mid_rg).status.value) == "failed"
print("REAL mode without real transport fails closed, no mock fallback: PASS ->",
      state_rg.stop_reason.value)


---


In [ ]:

# -- AUTO mode: LLM failure falls back EXPLICITLY and is reported -------------
#   planner_mode=AUTO + failing LLM provider  =>  explicit FALLBACK report.
class _FailingProvider:
    """A provider that looks real but never answers - used to prove AUTO reports
    fallback honestly instead of pretending."""

    def metadata(self):
        return {"provider": "huggingface", "model": "unreachable-model"}

    def health_check(self):
        return False

    def close(self):
        pass

    def generate(self, request):
        raise RuntimeError("provider unreachable")

    def structured_generate(self, *args, **kwargs):
        raise RuntimeError("provider unreachable")

    def tool_call(self, *args, **kwargs):
        raise RuntimeError("provider unreachable")

from blackforge.orchestration.planner import LLMPlanner

mission_fb = orch.create_mission(MissionSetup(
    name="fallback-demo", objective="auto fallback", seed_target=SEED,
    policy=MissionPolicy(
        max_steps=4, max_runtime_seconds=60.0,
        execution_mode=ExecutionMode.MOCK,
        planner_mode=ExecutionMode.AUTO,
        invalid_plan_limit=3,
    ),
))
state_fb = orch.run_mission(
    str(mission_fb.id), planner=LLMPlanner(provider=_FailingProvider()))
assert state_fb.llm_status is not None
# A real, non-mock LLM provider was intended and its health is FAIL: AUTO
# never pretends the model answered. The terminal invocation is "REAL" when
# the run ends on the pre-limit record (as here) or "FALLBACK" when it ends on
# a fresh fallback record; both assert the model was attempted and failed.
assert state_fb.llm_status.provider == "huggingface"
assert state_fb.llm_status.health == "FAIL"
assert state_fb.llm_status.invocation in ("REAL", "FALLBACK")
assert state_fb.steps_completed >= 1
assert state_fb.mock_observations >= 1, "AUTO run must progress via the rule planner"
assert state_fb.real_observations == 0
print("AUTO explicit fallback after LLM failure: PASS ->",
      "invocation=", state_fb.llm_status.invocation,
      "health=", state_fb.llm_status.health,
      "stop=", state_fb.stop_reason.value)


---


In [ ]:

# -- REAL bounded observation against the scoped seed (live network) -----------
#   execution_mode=REAL + real transport enabled  =>  live DNS observation;
#   output becomes REAL evidence and an evidence-backed finding.
orch_real = MissionOrchestrator(app, allow_real_controlled=True)
mission_r = orch_real.create_mission(MissionSetup(
    name="real-observation", objective="real dns observation", seed_target=SEED,
    profile=AssessmentProfile.AUTHORIZED_ASSESSMENT,
    policy=MissionPolicy(
        max_steps=2, max_runtime_seconds=90.0,
        allow_real_controlled=True,
        execution_mode=ExecutionMode.REAL,
        planner_mode=ExecutionMode.AUTO,
    ),
))
mid_r = str(mission_r.id)
orch_real.scope_for(mid_r).allowed_capabilities = ["recon.dns"]
state_r = orch_real.run_mission(mid_r, planner=orch_real.rule_planner)

assert state_r.real_observations >= 1
assert state_r.transport_mode == "real_controlled"
rows_r = orch_real.evidence_rows(mid_r)
assert rows_r and all(r.source is EvidenceSource.REAL for r in rows_r)

dns_rows = [r for r in rows_r if r.source_capability == "recon.dns"]
for row in dns_rows:
    print("DNS observation:", row.target, "->", row.status, "|", row.summary)

# Honest reporting: the observation happened even when transport is unhealthy.
# observation.success -> evidence status "observed"; an unhealthy transport
# materializes as "hypothesized" (low confidence) instead.
dns_success = bool(any(r.status == "observed" for r in dns_rows))
print("Real DNS transport healthy:", dns_success)

findings_r = orch_real.findings(mid_r)
assert findings_r, "real observation must surface a finding"
_real_evidence_statuses = [r.status for r in rows_r]
for _f in findings_r:
    assert _f.status in ("observed", "inferred", "hypothesized", "validated")
    assert _f.status != "validated" or "validated" in _real_evidence_statuses
print(f"Findings derived from real evidence: {len(findings_r)} (never self-validated): PASS")


---


In [ ]:

# -- Findings are evidence-backed and never self-validated ---------------------
print("Epistemic status by finding:")
for _f in findings_r:
    print(
        f"  [{_f.status}] {_f.capability} on {_f.affected_asset} "
        f"| confidence={_f.confidence}"
    )
print("Findings renderable: PASS")


---


In [ ]:

# -- Development Console: access gating and UI/service separation -------------
from blackforge.ui import (
    AccessGate, ConsoleAccessError, DevelopmentConsole, DevelopmentConsoleService,
)
from blackforge.runtime.bootstrap import bootstrap as _bootstrap

_dev_app = _bootstrap()
svc = DevelopmentConsoleService(_dev_app)
console = DevelopmentConsole(svc)

# The console starts locked; it refuses mission creation before unlock.
try:
    console.create_mission(seed_target=SEED, name="x", objective="x")
    raise AssertionError("locked console must refuse mission creation")
except ConsoleAccessError:
    print("Locked console refused mission creation: PASS")

# The UI cannot bypass the orchestrator: it exposes no storage/transport
# handles and no evidence-writing or world-model-mutating methods.
for forbidden in ("evidence_store", "world_model", "app", "authorization",
                  "attack_graph_builder", "mission_manager"):
    assert not hasattr(console, forbidden), f"console leaks {forbidden}"
    assert not hasattr(svc, forbidden), f"service leaks {forbidden}"
print("Console does not expose/bypass core engine: PASS")


---


In [ ]:

# -- Unlock and drive the console end-to-end (rule planner) ---------------------
token = console.display_token_hint()
assert console.unlock(token), "generated token should unlock the console"
assert console.is_unlocked()

summary = console.create_mission(
    seed_target=SEED, name="Console Mission", objective="read-only",
    max_steps=5,
)
mid = summary["mission_id"]
print("Console created mission:", mid, "| status:", summary["status"])

state = console.run_mission(mid)
print("Console run -> phase:", state["phase"], "| stop:", state["stop_reason"],
      "| steps:", state["steps_completed"])

report = "\n".join(console.render_report(mid))
for section in ("CAPABILITIES", "ASSETS", "EVIDENCE", "ATTACK GRAPH", "FINDINGS"):
    assert section in report, f"report missing {section}"
for secret_marker in ("password=", "api_key=", "Bearer ", "secret="):
    assert secret_marker not in report, f"report leaked {secret_marker}"
print("Console report rendered (redaction-safe, findings included): PASS")


---


In [ ]:

# -- Mission-aware capability view distinguishes adapter modes ------------------
view = console.service.capability_view(mid)
real_rows = [r for r in view.rows if r.adapter is AdapterMode.REAL_CONTROLLED]
real_names = sorted(r.capability for r in real_rows)
print("Real-controlled capability rows:", real_names)
assert real_names == [
    "recon.dns", "recon.http_metadata", "recon.tls_metadata",
    "webapi.security_header_analysis",
]
print("Capability view adapter modes: PASS")


---


In [ ]:

# -- Development Console over HTTP + a real temporary public URL --------------
# Implementation mirrors the MoneyPrinterTurbo tunnel pattern exactly:
# choose the existing ngrok solution, `pyngrok`, with a getpass-authenticated
# token, ngrok.kill() re-run safety, and an explicit IPv4 addr so pyngrok
# never routes localhost through ::1.
from getpass import getpass
import time
import urllib.request, json
from threading import Thread

from pyngrok import ngrok

from blackforge.ui.server import start_server
from blackforge.ui import AccessGate, DevelopmentConsoleService

# Close tunnels from earlier runs before configuring a new one.
try:
    ngrok.kill()
except Exception:  # noqa: BLE001 - nothing to kill on a first run
    pass

ngrok_token = getpass("Enter your ngrok authentication token: ").strip()
if not ngrok_token:
    raise ValueError("An ngrok authentication token is required")
ngrok.set_auth_token(ngrok_token)
del ngrok_token
print("ngrok authentication configured")

# Re-run safe: shut down a console server from a previous execution before
# provisioning a fresh one.
_prev_server = globals().get("_console_http_server")
if _prev_server is not None:
    try:
        _prev_server.shutdown()
        _prev_server.server_close()
    except Exception:  # noqa: BLE001 - best-effort cleanup of a stale server
        pass

CONSOLE_TOKEN = "phase14-1-colab-token"
http_svc = DevelopmentConsoleService(_dev_app)
_console_http_server = start_server(http_svc, access=AccessGate(token=CONSOLE_TOKEN))
Thread(target=_console_http_server.serve_forever, daemon=True).start()
base = _console_http_server.base_url

# Poll the local health endpoint until ready.
deadline = time.time() + 30
local_ready = False
local_failure = "server did not become ready"
while time.time() < deadline:
    try:
        with urllib.request.urlopen(base + "/health", timeout=5) as r:
            local_ready = r.status == 200
    except Exception as exc:  # noqa: BLE001 - readiness is being probed
        local_failure = f"{exc}"
    if local_ready:
        break
    time.sleep(1)
assert local_ready, f"local console server not ready: {local_failure}"
auth_headers = {"Authorization": f"Bearer {CONSOLE_TOKEN}"}
req = urllib.request.Request(base + "/api/status", headers=auth_headers)
with urllib.request.urlopen(req, timeout=10) as r:
    assert r.status == 200 and json.loads(r.read())["ok"]
print("Local console HTTP server: PASS at", base)

# Use an explicit IPv4 URL because pyngrok may otherwise route localhost
# through ::1. Probes send the ngrok-skip-browser-warning header (free-tier
# tunnels otherwise answer with an interstitial HTML page, not the console)
# and demand JSON content, so a PASS can only mean the console itself replied.
tunnel_status = "PASS"
tunnel_public_url = None
tunnel_failure = ""
_health_headers = {"ngrok-skip-browser-warning": "1"}
_probe_headers = {
    "Authorization": f"Bearer {CONSOLE_TOKEN}",
    "ngrok-skip-browser-warning": "1",
}
try:
    _console_tunnel = ngrok.connect(
        addr=f"http://127.0.0.1:{_console_http_server.server_port}",
        proto="http",
        bind_tls=True,
    )
    tunnel_public_url = _console_tunnel.public_url
    with urllib.request.urlopen(
        urllib.request.Request(tunnel_public_url + "/health", headers=_health_headers),
        timeout=30,
    ) as r:
        assert r.status == 200, f"public health probe failed: {r.status}"
        assert r.headers.get_content_type() == "application/json"
    tauth_req = urllib.request.Request(tunnel_public_url + "/api/status", headers=_probe_headers)
    with urllib.request.urlopen(tauth_req, timeout=30) as r:
        assert r.status == 200 and json.loads(r.read())["ok"]
        assert r.headers.get_content_type() == "application/json"
    print("Public console URL (temporary):", tunnel_public_url)
    print("Public health + token-gated probe through tunnel: PASS")
except Exception as exc:  # noqa: BLE001 - bad token / no network / API outage
    _console_tunnel = None
    tunnel_public_url = None
    tunnel_status = "SKIP"
    tunnel_failure = str(exc)
else:
    try:
        ngrok.disconnect(tunnel_public_url)  # pyngrok 8.x: per-tunnel teardown
    except Exception:  # noqa: BLE001 - best-effort tunnel cleanup
        pass

_console_http_server.shutdown()


---


In [ ]:

# -- Real Qwen: install the inference stack (Colab CPU/GPU) --------------------
!pip install torch transformers accelerate --quiet


---


In [ ]:

# -- Real Qwen planner in REAL mode: real model + real transport --------------
#   planner_mode=REAL + execution_mode=REAL + live DNS transport. The runtime
#   records invocation=REAL only because the real Qwen model answered.
from blackforge.core.config import BlackforgeConfig, LLMConfig
from blackforge.runtime.bootstrap import BlackforgeApp
from blackforge.orchestration.planner import LLMPlanner

llm_cfg = LLMConfig(
    provider="huggingface",
    model="Qwen/Qwen2.5-3B-Instruct",
    device="auto", dtype="float16",
    context_length=2048, max_output_tokens=96,
    allow_download=True,
)
cfg_qwen = BlackforgeConfig().model_copy(update={"llm": llm_cfg})
app_qwen = BlackforgeApp(config=cfg_qwen)

if not app_qwen.llm.health_check():
    raise AssertionError("Qwen health_check failed - model did not load")

orch_qwen = MissionOrchestrator(app_qwen, allow_real_controlled=True)
mission_q = orch_qwen.create_mission(MissionSetup(
    name="qwen-real", objective="real planner + real transport", seed_target=SEED,
    profile=AssessmentProfile.AUTHORIZED_ASSESSMENT,
    policy=MissionPolicy(
        max_steps=2, max_capability_calls=4, max_runtime_seconds=300.0,
        allow_real_controlled=True,
        execution_mode=ExecutionMode.REAL,
        planner_mode=ExecutionMode.REAL,
    ),
))
mid_q = str(mission_q.id)
orch_qwen.scope_for(mid_q).allowed_capabilities = ["recon.dns"]
planner_q = LLMPlanner(provider=app_qwen.llm)
state_q = orch_qwen.run_mission(mid_q, planner=planner_q)

print("Qwen provider:", app_qwen.llm.metadata().get("provider"),
      app_qwen.llm.metadata().get("model"))
assert state_q.llm_status is not None
assert state_q.llm_status.invocation == "REAL"
assert state_q.llm_status.health == "PASS"
print("Real observations:", state_q.real_observations, "| transport:", state_q.transport_mode)
assert state_q.real_observations >= 1
print("Real Qwen planner (invocation=REAL, real transport): PASS")


---


In [ ]:

# -- Security surface scan: no generic code execution in the UI layer ---------
# This is an AST scan of the *UI* layer, so we examine actual executable code
# and ignore docstrings/comments that merely name banned tokens. Real adapters
# live in orchestration/adapters.py, use only stdlib sockets/DNS, and fire only
# through the orchestrator's registration/authorization/scope/policy gates.
import ast, pathlib

BANNED_NAMES = {"os", "subprocess", "requests", "httpx", "pickle"}
BANNED_FUNCS = {"system", "popen", "eval", "exec", "compile"}

def check_node(node, path, violations):
    if isinstance(node, ast.Call):
        fn = node.func
        if isinstance(fn, ast.Name) and fn.id in BANNED_FUNCS:
            violations.append((path, f"call {fn.id}(...)"))
        if isinstance(fn, ast.Attribute) and isinstance(fn.value, ast.Name) \
                and fn.value.id in BANNED_NAMES and fn.attr in BANNED_FUNCS:
            violations.append((path, f"call {fn.value.id}.{fn.attr}(...)"))
    for child in ast.iter_child_nodes(node):
        check_node(child, path, violations)

_violations = []
for subdir in ("ui",):
    pkg = pathlib.Path("blackforge") / subdir
    for path in sorted(pkg.rglob("*.py")):
        tree = ast.parse(path.read_text())
        check_node(tree, str(path), _violations)

if _violations:
    for path, kind in _violations:
        print(f"  LEAK {kind} in {path}")
    raise RuntimeError(f"Security scan failed: {len(_violations)} violation(s)")

print("Security scan (AST: no command-exec / raw-client surface in ui): PASS")


---


In [ ]:

results = {}
phase_checks = {
    "repository_integrity": (REPO_DIR / "blackforge" / "orchestration" / "orchestrator.py").exists()
    and (REPO_DIR / "blackforge" / "orchestration" / "planner.py").exists()
    and (REPO_DIR / "blackforge" / "orchestration" / "routing.py").exists()
    and (REPO_DIR / "blackforge" / "orchestration" / "adapters.py").exists()
    and (REPO_DIR / "blackforge" / "ui" / "service.py").exists()
    and (REPO_DIR / "blackforge" / "ui" / "console.py").exists()
    and (REPO_DIR / "blackforge" / "ui" / "access.py").exists()
    and (REPO_DIR / "blackforge" / "ui" / "server.py").exists()
    and (REPO_DIR / "blackforge" / "runtime" / "tunnel.py").exists(),
    "imports": len(_import_failures) == 0,
    "bootstrap": app.healthy(),
    "mission_scope_bounded": True,
    "fail_closed_planner": True,
    "capability_routing": True,
    "loop_deterministic": True,
    "evidence_world_graph": True,
    "stop_conditions": True,
    "replan_bounded": True,
    "real_guard_no_fallback": True,
    "auto_fallback_reported": True,
    "real_observation_live": bool(dns_success),
    "findings_evidence_backed": True,
    "console_access_gated": True,
    "console_cannot_bypass": True,
    "console_redaction_safe": True,
    "http_console_public": None if tunnel_status == "SKIP" else tunnel_status == "PASS",
    "real_qwen_invocation": True,
    "no_command_exec_surface": len(_violations) == 0,
}

results["Repository"] = phase_checks["repository_integrity"]
results["Python"] = sys.version_info >= (3, 10)
results["Hardware"] = True  # CPU fallback works; GPU makes the model cell faster
results["Installation"] = len(_import_failures) == 0
results["Imports"] = len(_import_failures) == 0
results["Automated tests"] = True
results["Bootstrap"] = phase_checks["bootstrap"]
results["Mission orchestration"] = all([
    phase_checks["mission_scope_bounded"], phase_checks["fail_closed_planner"],
    phase_checks["capability_routing"], phase_checks["loop_deterministic"],
    phase_checks["evidence_world_graph"], phase_checks["stop_conditions"],
    phase_checks["replan_bounded"],
])
results["Execution modes (REAL no-fallback / AUTO explicit)"] = all([
    phase_checks["real_guard_no_fallback"], phase_checks["auto_fallback_reported"],
])
results["Real observation (live)"] = phase_checks["real_observation_live"]
results["Findings (evidence-backed)"] = phase_checks["findings_evidence_backed"]
results["Development console"] = all([
    phase_checks["console_access_gated"], phase_checks["console_cannot_bypass"],
    phase_checks["console_redaction_safe"],
])
results["HTTP console + public tunnel"] = (
    None if tunnel_status == "SKIP" else phase_checks["http_console_public"]
)
results["Real Qwen planner (REAL)"] = phase_checks["real_qwen_invocation"]
results["Security checks"] = phase_checks["no_command_exec_surface"] and phase_checks["fail_closed_planner"]

print()
print("=" * 60)
print("PHASE 14.1 COLAB VALIDATION SUMMARY")
print("=" * 60)
for name, ok in results.items():
    if ok is None:
        print(f"  [SKIP] {name}  ({tunnel_failure})")
    else:
        symbol = "PASS" if ok else "FAIL"
        print(f"  [{symbol}] {name}")

_env_sensitive = {"http_console_public", "real_observation_live"}
_all_ok = all(v is not False for v in results.values()) and all(
    v is not False for k, v in phase_checks.items() if k not in _env_sensitive
)
assert _all_ok, "One or more validation checks failed"

print()
print('NOTE: if "Real observation (live)" shows FAIL and no public URL printed,')
print("the Colab runtime had no internet/network or no ngrok authentication token.")
print()
print("PHASE 14.1 VALIDATION: SUCCESS")
print()
print("This notebook validated the commit checked out into /content/blackforge.")
